# Optimisation du Chiffre d'Affaires — Analyse Retail
## Étape 2 : Analyse Approfondie (Sellers & Temps)

Dans cette deuxième partie, je me concentre sur l'évolution du Chiffre d'Affaires (CA) au fil du temps et sur l'évaluation de la performance individuelle de mes vendeurs. DuckDB et Polars restent mes outils de prédilection.

In [ ]:
import polars as pl
import duckdb
import plotly.express as px

data_path = "../data/"

# Chargement rapide avec Polars
orders = pl.read_csv(data_path + "olist_orders_dataset.csv")
items = pl.read_csv(data_path + "olist_order_items_dataset.csv")
products = pl.read_csv(data_path + "olist_products_dataset.csv")
sellers = pl.read_csv(data_path + "olist_sellers_dataset.csv")

query = """
SELECT 
    i.order_id,
    date_trunc('month', CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS purchase_month,
    i.product_id,
    COALESCE(p.product_category_name, 'inconnue') AS category,
    i.seller_id,
    i.price
FROM items i
JOIN orders o ON i.order_id = o.order_id
LEFT JOIN products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
"""

df_master = duckdb.query(query).pl()


### 1. Analyse Temporelle du Chiffre d'Affaires
Je regarde comment notre CA global évolue mois après mois.

In [ ]:
timeline = df_master.group_by('purchase_month').agg(
    pl.sum('price').alias('total_ca')
).sort('purchase_month')

# Conversion en Pandas juste pour l'affichage avec Plotly
timeline_pd = timeline.to_pandas()
timeline_pd['purchase_month'] = timeline_pd['purchase_month'].dt.strftime('%Y-%m')

fig = px.line(
    timeline_pd, 
    x='purchase_month', 
    y='total_ca', 
    title="Évolution Mensuelle du Chiffre d'Affaires",
    markers=True
)
fig.show()

### 2. Vendeurs Sous-Performants
J'identifie les vendeurs dont le Chiffre d'Affaires total est inférieur à la moyenne des vendeurs de la plateforme.

In [ ]:
seller_analysis = df_master.group_by('seller_id').agg([
    pl.sum('price').alias('total_ca'),
    pl.len().alias('total_orders')
])

average_seller_ca = seller_analysis['total_ca'].mean()

bad_sellers = seller_analysis.filter(pl.col('total_ca') < average_seller_ca).sort('total_ca', descending=False)

print(f"Moyenne de CA par vendeur : ${average_seller_ca:.2f}")
print(f"Nombre de vendeurs sous la moyenne : {bad_sellers.shape[0]} / {seller_analysis.shape[0]}")
print("Top 5 des vendeurs les plus faibles :")
print(bad_sellers.head(5))

### 3. Relation Prix / Volume / CA
Y a-t-il un lien entre le fait de vendre beaucoup en volume et le chiffre d'affaires généré ?

In [ ]:
category_analysis = df_master.group_by('category').agg([
    pl.mean('price').alias('avg_price'),
    pl.len().alias('volume'),
    pl.sum('price').alias('total_ca')
])

fig = px.scatter(
    category_analysis.to_pandas(),
    x='volume',
    y='total_ca',
    size='avg_price',
    hover_name='category',
    title="Volume de ventes vs CA total par Catégorie (Taille = Prix Moyen)"
)
fig.show()

## Recommandations Business Concrètes

1. **Accompagnement des petits vendeurs** : La majorité de nos vendeurs réalisent un CA très inférieur à la moyenne. Nous devrions mettre en place un programme d'accompagnement (marketing, SEO interne) pour les aider à se développer.
2. **Ajustement de la Stratégie d'Acquisition** : Le graphique Volume vs CA montre que pousser les catégories à fort volume n'est pas toujours ce qui génère le plus de revenus globaux. Il faut allouer le budget marketing vers les catégories du cadran supérieur droit (fort volume, prix élevé).
3. **Saisonnalité** : Le Chiffre d'Affaires fluctue fortement selon les mois (avec un pic massif lors du Black Friday). Les ressources logistiques et le support client doivent être scalés précisément autour de ce mois critique.